In [ ]:
import os
import sys


!sudo apt-get update -qq
!sudo apt-get install -y libboost-all-dev ninja-build build-essential libgl1-mesa-dev > /dev/null

!pip uninstall -y torch torchvision torchaudio kaolin > /dev/null 2>&1
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121

print("📐 Installing NVIDIA Kaolin 0.17.0...")
!pip install kaolin==0.17.0 -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.4.0_cu121.html
if not os.path.exists("FoundationPose"):
    print("⬇️ Cloning FoundationPose...")
    !git clone https://github.com/NVlabs/FoundationPose.git
    !cd FoundationPose && git submodule update --init --recursive

%cd FoundationPose

# Overwrite requirements.txt to remove the broken version pins
print("📝 Writing clean requirements.txt...")
clean_requirements = """
hydra-core
iopath
transforms3d
trimesh
scikit-image
scipy
h5py
tqdm
moviepy
imageio
ninja
"""
with open('requirements.txt', 'w') as f:
    f.write(clean_requirements)

!pip install -r requirements.txt

print("🎨 Compiling nvdiffrast...")
if not os.path.exists("nvdiffrast"):
    !git clone https://github.com/NVlabs/nvdiffrast
!pip install ./nvdiffrast

print("⚙️ Compiling BundleSDF (C++)...")
os.makedirs("bundlesdf/build", exist_ok=True)
%cd bundlesdf/build
!rm -rf *
!cmake ..
!make -j$(nproc)
%cd ../..

Compile MyCUDA (The Python Bindings)
print("🔌 Compiling MyCUDA Bindings...")
%cd mycuda
os.environ['TORCH_CUDA_ARCH_LIST'] = "7.5;8.0;8.6;9.0" # Support all GPU types
!python setup.py build_ext --inplace
!python setup.py install
%cd ..

print("\n✅ NUCLEAR SETUP COMPLETE. You are ready to track.")

🚀 Initiating Manual Build Protocol...
📦 Installing system dependencies (Boost, Ninja, OpenGL)...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 116.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
🔥 Installing PyTorch 2.4.0...
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 19.3 MB/s eta 0:00:00
     ━━━━━━━━━

In [ ]:
# @title 🚀 Phase 2: Resume Build (Run AFTER Restarting Session)
import os
import sys
import torch

print(f"✅ Current Torch Version: {torch.__version__}")

# Sanity Check: Ensure we have the right version
if "2.4.0" not in torch.__version__:
    print("⚠️ WARNING: PyTorch 2.4 is NOT active. Did you run the Nuclear script first?")
else:
    print("🎉 Environment is correct! Proceeding to compilation...")

# --- Step 1: Force Clean the Broken Repo ---
# We delete the folder to ensure submodules (bundlesdf) are pulled correctly this time
if os.path.exists("FoundationPose"):
    print("🧹 Cleaning up broken installation...")
    !rm -rf FoundationPose

# --- Step 2: Re-Clone with Submodules ---
print("⬇️ Cloning FoundationPose (Fresh Copy)...")
!git clone https://github.com/NVlabs/FoundationPose.git
!cd FoundationPose && git submodule update --init --recursive

%cd FoundationPose

# --- Step 3: Install Requirements (Clean Version) ---
print("📝 Writing clean requirements.txt...")
clean_requirements = """
hydra-core
iopath
transforms3d
trimesh
scikit-image
scipy
h5py
tqdm
moviepy
imageio
ninja
"""
with open('requirements.txt', 'w') as f:
    f.write(clean_requirements)
!pip install -r requirements.txt

# --- Step 4: Compile Extensions (Now that Numpy is fixed) ---

# 4A: Nvdiffrast
print("🎨 Compiling nvdiffrast...")
if not os.path.exists("nvdiffrast"):
    !git clone https://github.com/NVlabs/nvdiffrast
!pip install ./nvdiffrast

# 4B: BundleSDF (The C++ Core)
# We use pip install . because cmake manual build is tricky with paths
print("⚙️ Compiling BundleSDF (This takes ~2-3 mins)...")
%cd bundlesdf
!pip install .
%cd ..

print("\n✅ BUILD COMPLETE. You can now run tracking.")

✅ Current Torch Version: 2.4.0+cu121
🎉 Environment is correct! Proceeding to compilation...
🧹 Cleaning up broken installation...
⬇️ Cloning FoundationPose (Fresh Copy)...
Cloning into 'FoundationPose'...
remote: Enumerating objects: 233, done.
remote: Total 233 (delta 0), reused 0 (delta 0), pack-reused 233 (from 1)
Receiving objects: 100% (233/233), 120.80 MiB | 41.77 MiB/s, done.
Resolving deltas: 100% (64/64), done.
/content/FoundationPose
📝 Writing clean requirements.txt...
  Using cached numpy-2.4.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.4.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but y

In [ ]:
# @title 🚀 Phase 3: Final Compilation Fix (Run This)
import os
import sys

print("🔧 Starting Final Compilation Phase...")

# Ensure we are in the right place
%cd /content/FoundationPose

# --- Fix 1: Compile Nvdiffrast (Bypassing Build Isolation) ---
print("🎨 Compiling Nvdiffrast (Force System Tools)...")
if not os.path.exists("nvdiffrast"):
    !git clone https://github.com/NVlabs/nvdiffrast

# We use --no-build-isolation so it finds the Ninja/OpenGL we installed earlier
!pip install ./nvdiffrast --no-build-isolation

# --- Fix 2: Compile BundleSDF (The C++ Way) ---
print("⚙️ Compiling BundleSDF (C++ Core)...")
# We explicitly use CMake because this is NOT a python package
os.makedirs("bundlesdf/build", exist_ok=True)
%cd bundlesdf/build
!rm -rf * # Clean any failed states
!cmake ..
!make -j$(nproc)
%cd ../..

# --- Fix 3: Compile MyCUDA (Python Bindings) ---
print("🔌 Compiling MyCUDA Bindings...")
# This IS a python package, but we build it in-place
if os.path.exists("mycuda"):
    %cd mycuda
    os.environ['TORCH_CUDA_ARCH_LIST'] = "7.5;8.0;8.6;9.0"
    !python setup.py build_ext --inplace
    !python setup.py install
    %cd ..
else:
    print("⚠️ 'mycuda' folder not found! Checking inside bundlesdf...")
    # Sometimes it lives inside bundlesdf depending on the commit
    if os.path.exists("bundlesdf/mycuda"):
        %cd bundlesdf/mycuda
        os.environ['TORCH_CUDA_ARCH_LIST'] = "7.5;8.0;8.6;9.0"
        !python setup.py build_ext --inplace
        !python setup.py install
        %cd ../..

print("\n✅ FINAL CHECK: If you see no red errors above, you are done!")

🔧 Starting Final Compilation Phase...
/content/FoundationPose
🎨 Compiling Nvdiffrast (Force System Tools)...
Processing ./nvdiffrast
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-2.4.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.4.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
  Created wheel for nvdiffrast: filename=nvdiffrast-0.4.0-cp312-cp312-linux_x86_64.whl size=16493414 sha256=6a51daf959ebd2606a0b82c9ede6a4027575329d334e099eb9f337aebb618105
  Stored in directory: /tmp/pip-ephem-wheel-cache-nds1j6cv/wheels/3a/8c/41/02c5be20df3b626ed903d0ec14cdd134c2294b1c0a0fcccf85
Successfully built nvdiffrast
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
numba 0.60.0 requir

In [ ]:
# @title 🚀 Phase 4: Fix Missing Headers & C++ Version
import os
import sys

print("🔧 Applying Final Patches...")

# --- Fix 1: Install Eigen (The missing math library) ---
print("📦 Installing Eigen3...")
!sudo apt-get install -y libeigen3-dev > /dev/null
# Symlink it so the compiler finds it easily
!ln -sf /usr/include/eigen3/Eigen /usr/include/Eigen

# --- Fix 2: Patch setup.py for C++17 ---
print("📝 Patching setup.py to force C++17...")
%cd /content/FoundationPose/bundlesdf/mycuda

# We read the setup.py and replace c++14 with c++17
with open("setup.py", "r") as f:
    setup_content = f.read()

if "c++14" in setup_content:
    print("  - Found C++14 flag, upgrading to C++17...")
    setup_content = setup_content.replace("c++14", "c++17")
    with open("setup.py", "w") as f:
        f.write(setup_content)
else:
    print("  - setup.py already seems patched or doesn't have explicit flags.")

# --- Fix 3: Re-Run Compilation ---
print("🔌 Re-compiling MyCUDA Bindings...")
# We add the global include path for Eigen just in case
os.environ['CPLUS_INCLUDE_PATH'] = "/usr/include/eigen3"
os.environ['TORCH_CUDA_ARCH_LIST'] = "7.5;8.0;8.6;9.0"

!python setup.py build_ext --inplace
!python setup.py install

print("\n✅ DONE. You should see 'Finished processing dependencies' above.")

🔧 Applying Final Patches...
📦 Installing Eigen3...
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
📝 Patching setup.py to force C++17...
/content/FoundationPose/bundlesdf/mycuda
  - Found C++14 flag, upgrading to C++17...
🔌 Re-compiling MyCUDA Bindings...
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/dist.py:261: UserWarning: Unknown distribution option: 'extra_cflags'
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/dist.py:261: UserWarning: Unknown distribution option: 'extra_cuda_cflags'
  warnings.warn(msg)
running build_ext
/usr/local/lib

In [ ]:
# @title 🚀 "Hello World" Test (Mustard Bottle)
import os

%cd /content/FoundationPose

# --- 1. Download Weights (Required) ---
if not os.path.exists("weights"):
    print("⬇️ Downloading Weights...")
    !pip install gdown
    !gdown --id 1s4pB6p4ApfWMiMjmTXOFco8dHbNXikp- -O weights.zip
    !unzip -q -o weights.zip
    print("✅ Weights Ready.")

# --- 2. Download Sample Data (Mustard Bottle) ---
# This is the official demo data from NVLabs
if not os.path.exists("demo_data"):
    print("⬇️ Downloading Sample Data...")
    !gdown --id 1v63_207d6i3Qo88X8n_hE_u62i4oD1- -O demo_data.zip
    !unzip -q -o demo_data.zip
    print("✅ Sample Data Ready.")

# --- 3. Run the Demo ---
# We use 'model_based' because we have the CAD file for the mustard bottle
mesh_path = "demo_data/mustard0/mesh/textured.obj"
video_path = "demo_data/mustard0/mustard0.mp4"

print(f"🎥 Running 'Hello World' on: {video_path}")

!python run_demo.py \
  --mode model_based \
  --mesh_path {mesh_path} \
  --video_path {video_path} \
  --output_dir debug_output \
  --debug 1

print("\n✅ Hello World Complete! Check 'debug_output' folder for visualization.")

/content/FoundationPose
⬇️ Downloading Weights...
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/common-0.0.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gdown/download.py:33: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into 

In [ ]:
# @title 🛠️ Fix Missing PyTorch3D
import os
import sys

print("🔧 Installing PyTorch3D (This takes ~5 minutes)...")

# 1. Install dependencies
!pip install fvcore iopath

# 2. Build PyTorch3D from source
# We force it to use the new C++17 standard we set up earlier
os.environ['CPLUS_INCLUDE_PATH'] = "/usr/include/eigen3"
os.environ['TORCH_CUDA_ARCH_LIST'] = "7.5;8.0;8.6;9.0"
!pip install "git+https://github.com/facebookresearch/pytorch3d.git"

print("✅ PyTorch3D Installed.")

🔧 Installing PyTorch3D (This takes ~5 minutes)...
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/common-0.0.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=4995fc95bfa80c5c51e970e9a22122c85c8f548100c44954704122d0e1ae3d67
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
Successfully built fvcore
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/common-0.0.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussio

In [ ]:
# @title 🛠️ Fix Weights Placement
import os
import shutil

# 1. Create the destination folder
base_path = "/content/FoundationPose"
weights_dir = os.path.join(base_path, "weights")
os.makedirs(weights_dir, exist_ok=True)

# 2. Find your uploaded zip
zip_path = os.path.join(base_path, "weights.zip")

if os.path.exists(zip_path):
    print("📦 Found weights.zip! Unpacking...")
    # Unzip directly into the weights folder
    !unzip -q -o "{zip_path}" -d "{weights_dir}"

    # 3. Smart Fix: If you zipped a parent folder by mistake, move items up
    # We check if the folders are nested too deep (e.g. weights/weights/2023...)
    for root, dirs, files in os.walk(weights_dir):
        for d in dirs:
            if d.startswith("2023") or d.startswith("2024"):
                src = os.path.join(root, d)
                dst = os.path.join(weights_dir, d)
                if src != dst:
                    print(f"   Moving {d} to correct location...")
                    shutil.move(src, dst)

    print("✅ Weights placed correctly in /content/FoundationPose/weights/")
else:
    print(f"❌ Could not find '{zip_path}'")
    print("👉 Please drag 'weights.zip' into the 'FoundationPose' folder in the sidebar.")

📦 Found weights.zip! Unpacking...
   Moving 2024-01-11-20-02-45 to correct location...
   Moving 2023-10-28-18-33-37 to correct location...
   Moving 2024-01-11-20-02-45 to correct location...


Error: Destination path '/content/FoundationPose/weights/2024-01-11-20-02-45/2024-01-11-20-02-45' already exists

In [ ]:
# @title 🛠️ Smart Fix: Flatten Folder Structure
import os
import shutil

base_path = "/content/FoundationPose/weights"
required_folders = ["2023-10-28-18-33-37", "2024-01-11-20-02-45"]

print("🧹 Scanning for weight files...")

# Find all 'model_best.pth' files anywhere in the weights folder
for root, dirs, files in os.walk(base_path):
    for file in files:
        if file == "model_best.pth":
            # This is a valid model file. Get its parent folder name.
            parent_folder_name = os.path.basename(root)
            current_location = os.path.join(root, file)

            # Target location: /content/FoundationPose/weights/FOLDER_NAME/model_best.pth
            target_dir = os.path.join(base_path, parent_folder_name)
            target_file = os.path.join(target_dir, "model_best.pth")

            # If it's already in the right place, skip
            if current_location == target_file:
                continue

            print(f"   Found deep file at: {current_location}")
            print(f"   Moving to: {target_file}")

            # Move and overwrite
            os.makedirs(target_dir, exist_ok=True)
            shutil.move(current_location, target_file)

# Cleanup empty nested folders
print("✨ Cleaning up empty folders...")
for root, dirs, files in os.walk(base_path, topdown=False):
    for d in dirs:
        try:
            os.rmdir(os.path.join(root, d))
        except OSError:
            pass # Folder not empty

# Verify
print("\n🔍 Final Verification:")
all_good = True
for folder in required_folders:
    if os.path.exists(os.path.join(base_path, folder, "model_best.pth")):
        print(f"   ✅ {folder}: OK")
    else:
        print(f"   ❌ {folder}: MISSING")
        all_good = False

if all_good:
    print("\n🚀 STRUCTURE CORRECT. You can run the demo below!")
else:
    print("\n⚠️ Still missing files. Please check your zip file content.")

🧹 Scanning for weight files...
✨ Cleaning up empty folders...

🔍 Final Verification:
   ✅ 2023-10-28-18-33-37: OK
   ✅ 2024-01-11-20-02-45: OK

🚀 STRUCTURE CORRECT. You can run the demo below!


In [ ]:
# @title 🚀 Run Hello World
%cd /content/FoundationPose

# Download demo data if missing
if not os.path.exists("demo_data"):
    print("⬇️ Downloading Demo Data...")
    !pip install gdown > /dev/null
    !gdown --id 1v63_207d6i3Qo88X8n_hE_u62i4oD1- -O demo_data.zip
    !unzip -q -o demo_data.zip

print("\n🎥 RUNNING TRACKER...")
!python run_demo.py \
  --mode model_based \
  --mesh_path "demo_data/mustard0/mesh/textured.obj" \
  --video_path "demo_data/mustard0/mustard0.mp4" \
  --output_dir debug_output \
  --debug 1

print("\n✅ Check 'debug_output' folder for results!")

/content/FoundationPose
⬇️ Downloading Demo Data...
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/common-0.0.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1v63_207d6i3Qo88X8n_hE_u62i4oD1-

but Gdown can't. Please check connections and permissions.
unzi

In [ ]:
# @title 🛠️ Install Open3D Dependency
print("📦 Installing Open3D...")
!pip install open3d
print("✅ Open3D Installed.")

📦 Installing Open3D...
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/common-0.0.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 142.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 98.4 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-

In [ ]:
%cd /content/FoundationPose/demo_data
!unzip -q mustard0.zip

/content/FoundationPose/demo_data


In [ ]:
# @title 🛠️ Install Missing Math Library
print("📦 Installing transformations library...")
!pip install transformations
print("✅ Library installed.")

📦 Installing transformations library...
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/common-0.0.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.3/149.3 kB 15.0 MB/s eta 0:00:00
✅ Library installed.


In [ ]:
# @title 🛠️ Install Configuration Parser
print("📦 Installing ruamel.yaml...")
!pip install ruamel.yaml
print("✅ Library installed.")

📦 Installing ruamel.yaml...
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/common-0.0.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 11.9 MB/s eta 0:00:00
✅ Library installed.


In [ ]:
# @title 🛠️ Install Kornia Dependency
print("📦 Installing Kornia...")
!pip install kornia
print("✅ Kornia Installed.")

📦 Installing Kornia...
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/common-0.0.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 94.5 MB/s eta 0:00:00
✅ Kornia Installed.


In [ ]:
%cd /content/FoundationPose

# Running with arguments that match your script's 'usage' definitions
!python run_demo.py \
  --mesh_file demo_data/mustard0/mesh/textured.obj \
  --test_scene_dir demo_data/mustard0 \
  --debug 1 \
  --debug_dir debug_output

/content/FoundationPose
Warp 1.11.0 initialized:
   CUDA Toolkit 12.9, Driver 12.4
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.11.0
Traceback (most recent call last):
  File "/content/FoundationPose/run_demo.py", line 29, in <module>
    mesh = trimesh.load(args.mesh_file)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/trimesh/exchange/load.py", line 111, in load
    loaded = load_scene(
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/trimesh/exchange/load.py", line 193, in load_scene
    arg = _parse_file_args(
          ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/trimesh/exchange/load.py", line 624, in _parse_file_args
    raise ValueError(f"string is not a file: `{file_obj}`")
ValueError: string is not a file: `demo_data/mustard0/mesh/textured.obj`


In [ ]:
# @title 🔍 Find the Mesh File
!find /content/FoundationPose/demo_data -name "textured.obj"

In [ ]:
%cd /content/FoundationPose

# Using absolute paths is the safest way in Colab
!python run_demo.py

/content/FoundationPose
Warp 1.11.0 initialized:
   CUDA Toolkit 12.9, Driver 12.4
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.11.0
[__init__()] self.cfg: 
 lr: 0.0001
c_in: 6
zfar: 'Infinity'
debug: null
n_view: 1
run_id: 3wy8qqex
use_BN: true
exp_name: 2024-01-11-20-02-45
n_epochs: 62
save_dir: /home/bowenw/debug/2024-01-11-20-02-45/
use_mask: false
loss_type: pairwise_valid
optimizer: adam
batch_size: 64
crop_ratio: 1.1
enable_amp: true
use_normal: false
max_num_key: null
warmup_step: -1
input_resize:
- 160
- 160
max_step_val: 1000
vis_interval: 1000
weight_decay: 0
normalize_xyz: true
resume_run_id: null
clip_grad_norm: 'Infinity'
lr_epoch_decay: 500
render_backend: nvdiffrast
train_num_pair: 5
lr_decay_epochs:
- 50
n_epochs_warmup: 1
make_pair_online: false
gradient_max_norm: 'Infinity'
max_step_per_epoch: 10000
n_rendering_workers: 1
save_epoch_interval: 100
n_dataloade

In [ ]:
# @title 🛠️ Fix mycpp WITHOUT changing Python version
import os
import pybind11

# 1. Patch the specific line causing the 3.12 error
cpp_file = "/content/FoundationPose/mycpp/src/app/pybind_api.cpp"
if os.path.exists(cpp_file):
    with open(cpp_file, 'r') as f:
        lines = f.readlines()
    with open(cpp_file, 'w') as f:
        for line in lines:
            # We comment out the line that Python 3.12 dislikes
            if "frame = frame->f_back;" in line:
                f.write("// " + line)
            else:
                f.write(line)
    print("✅ Fixed pybind_api.cpp for Python 3.12")

# 2. Re-compile ONLY the missing mycpp module
%cd /content/FoundationPose/mycpp
os.makedirs("build", exist_ok=True)
%cd build
!cmake .. -Dpybind11_DIR={pybind11.get_cmake_dir()}
!make -j$(nproc)
!cp *.so ..
print("🎉 mycpp is now compatible and compiled!")

✅ Fixed pybind_api.cpp for Python 3.12
/content/FoundationPose/mycpp
/content/FoundationPose/mycpp/build
CMake Warning (dev) at CMakeLists.txt:10 (find_package):
  Policy CMP0167 is not set: The FindBoost module is removed.  Run "cmake
  --help-policy CMP0167" for policy details.  Use the cmake_policy command to
  set the policy and suppress this warning.

This warning is for project developers.  Use -Wno-dev to suppress it.

CMake Warning (dev) at /usr/local/lib/python3.12/dist-packages/pybind11/share/cmake/pybind11/FindPythonLibsNew.cmake:101 (message):
  Policy CMP0148 is not set: The FindPythonInterp and FindPythonLibs modules
  are removed.  Run "cmake --help-policy CMP0148" for policy details.  Use
  the cmake_policy command to set the policy and suppress this warning, or
  preferably upgrade to using FindPython, either by calling it explicitly
  before pybind11, or by setting PYBIND11_FINDPYTHON ON before pybind11.
Call Stack (most recent call first):
  /usr/local/lib/python3.12

In [ ]:
# @title 🚀 Run the FoundationPose Tracker
import os
import sys

# 1. Ensure Python can see your new builds
paths = ['/content/FoundationPose', '/content/FoundationPose/mycpp', '/content/FoundationPose/mycuda']
for p in paths:
    if p not in sys.path:
        sys.path.append(p)

%cd /content/FoundationPose

# 2. Trigger the Demo
# We use 'model_based' which uses the .obj file to track
!python run_demo.py \
  --mesh_file /content/FoundationPose/demo_data/mustard0/mesh/textured.obj \
  --test_scene_dir /content/FoundationPose/demo_data/mustard0 \
  --debug 1 \
  --debug_dir debug_output

print("\n---------------------------------------------------")
if os.path.exists("debug_output/mustard0/ob_1.png"):
    print("🎉 SUCCESS! Tracking complete.")
    print("Check the '/content/FoundationPose/debug_output/mustard0' folder for images.")
else:
    print("❌ Tracker finished, but no output images found.")

/content/FoundationPose
Warp 1.11.0 initialized:
   CUDA Toolkit 12.9, Driver 12.4
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.11.0
Traceback (most recent call last):
  File "/content/FoundationPose/run_demo.py", line 29, in <module>
    mesh = trimesh.load(args.mesh_file)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/trimesh/exchange/load.py", line 111, in load
    loaded = load_scene(
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/trimesh/exchange/load.py", line 193, in load_scene
    arg = _parse_file_args(
          ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/trimesh/exchange/load.py", line 624, in _parse_file_args
    raise ValueError(f"string is not a file: `{file_obj}`")
ValueError: string is not a file: `/content/FoundationPose/demo_data/mustard0/mesh/textured.obj`

-------------

In [ ]:
# @title 🛠️ Sanity Check: Reset Data & Run
import os

# 1. Force a clean extraction of demo data
%cd /content/FoundationPose
if os.path.exists("demo_data.zip"):
    print("📦 Extracting demo data to a clean folder...")
    !unzip -q -o demo_data.zip -d /content/sanity_data
else:
    print("❌ demo_data.zip not found! Please ensure it's in /content/FoundationPose")

# 2. Define the absolute paths based on the standard zip structure
# Most FoundationPose zips put the mesh in: mustard0/mesh/textured.obj
mesh_path = "/content/sanity_data/mustard0/mesh/textured.obj"
scene_dir = "/content/sanity_data/mustard0"

# 3. Final Run Command
if os.path.exists(mesh_path):
    print("✅ Sanity data located. Launching A100...")
    !python run_demo.py \
      --mesh_file {mesh_path} \
      --test_scene_dir {scene_dir} \
      --debug 1 \
      --debug_dir /content/FoundationPose/debug_output
else:
    print("❌ Path check failed. Let's list the folder to see what happened:")
    !ls -R /content/sanity_data | head -n 20

/content/FoundationPose
❌ demo_data.zip not found! Please ensure it's in /content/FoundationPose
❌ Path check failed. Let's list the folder to see what happened:
ls: cannot access '/content/sanity_data': No such file or directory


In [ ]:
# @title 🚀 Simplest Sanity Demo
import os

# 1. Define paths based on your EXACT folder structure in image_b4ca27.png
# Note: Your mesh file is named 'textured_simple.obj', NOT 'textured.obj'
mesh_file = "/content/FoundationPose/demo_data/mustard0/mesh/textured_simple.obj"
test_scene = "/content/FoundationPose/demo_data/mustard0"
debug_out = "/content/FoundationPose/debug_output"

# 2. Run the command using the 'usage' arguments from your previous error
%cd /content/FoundationPose

if os.path.exists(mesh_file):
    print("✅ Found mesh_file. Launching tracking...")
    !python run_demo.py \
      --mesh_file {mesh_file} \
      --test_scene_dir {test_scene} \
      --debug 1 \
      --debug_dir {debug_out}
else:
    print(f"❌ ERROR: Cannot find {mesh_file}")
    print("Please verify the filename in the 'mesh' folder in your sidebar.")

/content/FoundationPose
✅ Found mesh_file. Launching tracking...
Warp 1.11.0 initialized:
   CUDA Toolkit 12.9, Driver 12.4
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.11.0
[__init__()] self.cfg: 
 lr: 0.0001
c_in: 6
zfar: 'Infinity'
debug: null
n_view: 1
run_id: 3wy8qqex
use_BN: true
exp_name: 2024-01-11-20-02-45
n_epochs: 62
save_dir: /home/bowenw/debug/2024-01-11-20-02-45/
use_mask: false
loss_type: pairwise_valid
optimizer: adam
batch_size: 64
crop_ratio: 1.1
enable_amp: true
use_normal: false
max_num_key: null
warmup_step: -1
input_resize:
- 160
- 160
max_step_val: 1000
vis_interval: 1000
weight_decay: 0
normalize_xyz: true
resume_run_id: null
clip_grad_norm: 'Infinity'
lr_epoch_decay: 500
render_backend: nvdiffrast
train_num_pair: 5
lr_decay_epochs:
- 50
n_epochs_warmup: 1
make_pair_online: false
gradient_max_norm: 'Infinity'
max_step_per_epoch: 10000
n_rendering_worker

In [ ]:
# @title 🩹 Patch run_demo.py for Colab
import os

file_path = "/content/FoundationPose/run_demo.py"
with open(file_path, 'r') as f:
    lines = f.readlines()

with open(file_path, 'w') as f:
    for line in lines:
        # Disable GUI window functions that crash in Colab
        if "cv2.imshow" in line or "cv2.waitKey" in line:
            f.write(f"    # {line.strip()}\n")
        else:
            f.write(line)

print("✅ Patched run_demo.py. GUI windows disabled.")

✅ Patched run_demo.py. GUI windows disabled.


In [ ]:
%cd /content/FoundationPose

!python run_demo.py \
  --mesh_file /content/FoundationPose/demo_data/mustard0/mesh/textured_simple.obj \
  --test_scene_dir /content/FoundationPose/demo_data/mustard0 \
  --debug 1 \
  --debug_dir debug_output

Streaming output truncated to the last 5000 lines.
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[track_one()] pose done
[<module>()] i:520
[track_one()] Welcome
[track_one()] depth processing done
[predict()] ob_in_cams:(1, 4, 4)
[predict()] self.cfg.use_normal:False
[predict()] trans_normalizer:[0.019999999552965164, 0.019999999552965164, 0.05000000074505806], rot_normalizer:0.3490658503988659
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_

In [ ]:
# @title 💾 Permanent Backup to Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create a permanent backup folder
!mkdir -p /content/drive/MyDrive/FoundationPose_Sanity_Backup

# Zip the compiled .so files, build folders, and neural weights
print("📦 Archiving your infra work...")
!zip -r /content/drive/MyDrive/FoundationPose_Sanity_Backup/infra_compiled.zip \
    /content/FoundationPose/mycpp/*.so \
    /content/FoundationPose/mycuda/build \
    /content/FoundationPose/bundlesdf/build \
    /content/FoundationPose/weights

print("✅ Backup Complete. You can now safely close Colab.")

Mounted at /content/drive
📦 Archiving your infra work...
	zip warning: name not matched: /content/FoundationPose/mycuda/build
  adding: content/FoundationPose/mycpp/mycpp.cpython-312-x86_64-linux-gnu.so (deflated 57%)
  adding: content/FoundationPose/bundlesdf/build/ (stored 0%)
  adding: content/FoundationPose/weights/ (stored 0%)
  adding: content/FoundationPose/weights/__MACOSX/ (stored 0%)
  adding: content/FoundationPose/weights/__MACOSX/._2024-01-11-20-02-45 (deflated 36%)
  adding: content/FoundationPose/weights/__MACOSX/._2023-10-28-18-33-37 (deflated 36%)
  adding: content/FoundationPose/weights/2024-01-11-20-02-45/ (stored 0%)
  adding: content/FoundationPose/weights/2024-01-11-20-02-45/config.yml (deflated 44%)
  adding: content/FoundationPose/weights/2024-01-11-20-02-45/model_best.pth (deflated 8%)
  adding: content/FoundationPose/weights/2024-01-11-20-02-45/2024-01-11-20-02-45/ (stored 0%)
  adding: content/FoundationPose/weights/2024-01-11-20-02-45/2024-01-11-20-02-45/._m

In [ ]:
# @title 🔗 Connect to EgoDex Data
from google.colab import drive
import os
drive.mount('/content/drive')

# Pick one of your subdirectories (e.g., open_close)
VIDEO_DIR = "/content/drive/MyDrive/EgoDex_Data/open_close"
REF_DIR = "/content/FoundationPose/ref_views_box"

os.makedirs(REF_DIR, exist_ok=True)
os.makedirs(f"{REF_DIR}/rgb", exist_ok=True)
os.makedirs(f"{REF_DIR}/depth", exist_ok=True)
os.makedirs(f"{REF_DIR}/masks", exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# @title 📸 Sample 16 Views & Prep Intrinsics
import glob
import shutil
import random

# Get your frames
rgb_files = sorted(glob.glob(f"{VIDEO_DIR}/rgb/*.png"))
depth_files = sorted(glob.glob(f"{VIDEO_DIR}/depth/*.png"))

# Sample 16 random views
sample_indices = random.sample(range(len(rgb_files)), 16)
for i, idx in enumerate(sample_indices):
    shutil.copy(rgb_files[idx], f"{REF_DIR}/rgb/{i:06d}.png")
    shutil.copy(depth_files[idx], f"{REF_DIR}/depth/{i:06d}.png")
    # Note: Ensure you have corresponding masks in {REF_DIR}/masks/

# Copy your camera intrinsics (Mandatory)
shutil.copy(f"{VIDEO_DIR}/cam_K.txt", f"{REF_DIR}/cam_K.txt")

ValueError: Sample larger than population or is negative

In [ ]:
# @title 📽️ Extract 16 Random Reference Views from MP4
import cv2
import os
import random

# --- Configuration ---
VIDEO_PATH = "/content/drive/MyDrive/EgoDex_Data/input_video/video_learning_samples/open_close/1.mp4"
REF_DIR = "/content/FoundationPose/ref_views_box"

# 1. Setup Directories
for sub in ['rgb', 'depth', 'masks']:
    os.makedirs(os.path.join(REF_DIR, sub), exist_ok=True)

# 2. Open Video File
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    print("❌ Error: Could not open video file.")
else:
    # 3. Get total frames and sample indices
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"🎞️ Video has {total_frames} total frames.")

    # Sample 16 unique random frame indices
    sample_indices = sorted(random.sample(range(total_frames), 16))

    for i, frame_idx in enumerate(sample_indices):
        # Jump to the specific frame
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()

        if ret:
            # Save the frame as an RGB reference
            filename = f"{i:06d}.png"
            cv2.imwrite(os.path.join(REF_DIR, "rgb", filename), frame)

            # NOTE: For model-free, you still need the corresponding
            # Depth maps for these 16 frames in REF_DIR/depth/
            print(f"✅ Saved frame {frame_idx} as ref {i}")
        else:
            print(f"⚠️ Failed to read frame {frame_idx}")

    cap.release()
    print(f"🎉 Done! 16 views extracted to {REF_DIR}/rgb")

🎞️ Video has 171 total frames.
✅ Saved frame 1 as ref 0
✅ Saved frame 7 as ref 1
✅ Saved frame 25 as ref 2
✅ Saved frame 30 as ref 3
✅ Saved frame 36 as ref 4
✅ Saved frame 56 as ref 5
✅ Saved frame 61 as ref 6
✅ Saved frame 62 as ref 7
✅ Saved frame 109 as ref 8
✅ Saved frame 110 as ref 9
✅ Saved frame 111 as ref 10
✅ Saved frame 123 as ref 11
✅ Saved frame 147 as ref 12
✅ Saved frame 148 as ref 13
✅ Saved frame 163 as ref 14
✅ Saved frame 168 as ref 15
🎉 Done! 16 views extracted to /content/FoundationPose/ref_views_box/rgb


In [ ]:
# @title 🛠️ Final Prep: Depth Conversion & Intrinsics
import os
import cv2
import numpy as np

# --- 1. USER CONFIGURATION ---
# Path where your Depth Anything .npy or relative images are stored
RELATIVE_DEPTH_DIR = "/content/drive/MyDrive/EgoDex_Data/open_close/depth_anything_output"
REF_DIR = "/content/FoundationPose/ref_views_box"

# Camera Intrinsics for your EgoDex device (Change these to your real values!)
# Format: [fx, 0, cx], [0, fy, cy], [0, 0, 1]
K_MATRIX = np.array([
    [910.0, 0.0, 960.0],  # fx, 0, cx
    [0.0, 910.0, 540.0],  # 0, fy, cy
    [0.0, 0.0, 1.0]
])

# --- 2. GENERATE cam_K.txt ---
# FoundationPose expects a simple space-separated 3x3 matrix in a .txt file
with open(os.path.join(REF_DIR, "cam_K.txt"), "w") as f:
    for row in K_MATRIX:
        f.write(" ".join(map(str, row)) + "\n")
print("✅ Generated cam_K.txt")

# --- 3. CONVERT DEPTH TO METRIC ---
# We convert relative depth to 16-bit metric depth (millimeters or scaled meters)
# FoundationPose typically handles depth as a 1D BxHxW tensor of float values
print("🔄 Converting 16 reference depth maps...")
for i in range(16):
    # Load your relative depth (adjust extension if needed)
    rel_depth_path = os.path.join(RELATIVE_DEPTH_DIR, f"{i:06d}.png")
    if not os.path.exists(rel_depth_path): continue

    rel_depth = cv2.imread(rel_depth_path, cv2.IMREAD_UNCHANGED)

    # CRITICAL: Scale factor to convert 'relative' to 'meters'
    # If your box is ~0.5m away and pixel value is 128, scale is 0.5/128
    SCALE_FACTOR = 0.001
    metric_depth = rel_depth.astype(np.float32) * SCALE_FACTOR

    # Save as .npy or 16-bit .png as per FoundationPose requirements
    np.save(os.path.join(REF_DIR, "depth", f"{i:06d}.npy"), metric_depth)

print(f"🎉 Done! Ready to run: !python bundlesdf/run_nerf.py --ref_view_dir {REF_DIR} --dataset custom_box")

✅ Generated cam_K.txt
🔄 Converting 16 reference depth maps...
🎉 Done! Ready to run: !python bundlesdf/run_nerf.py --ref_view_dir /content/FoundationPose/ref_views_box --dataset custom_box


In [ ]:
!python bundlesdf/run_nerf.py --ref_view_dir /content/FoundationPose/ref_views_box --dataset custom_box

Warp 1.11.0 initialized:
   CUDA Toolkit 12.9, Driver 12.4
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.11.0
Traceback (most recent call last):
  File "/content/FoundationPose/bundlesdf/run_nerf.py", line 115, in <module>
    run_linemod()
  File "/content/FoundationPose/bundlesdf/run_nerf.py", line 98, in run_linemod
    mesh = run_one_ob(base_dir=base_dir, cfg=cfg, use_refined_mask=True)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/FoundationPose/bundlesdf/run_nerf.py", line 52, in run_one_ob
    with open(f'{base_dir}/select_frames.yml','r') as ff:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/FoundationPose/ref_views_box/ob_0000001/select_frames.yml'
